# Chicago 311 Service Requests - MongoDB Analiza

**Autor:** Marko Zelić IN48-2022  
**Predmet:** Sistemi baza podataka, FTN Novi Sad  
**Dataset:** Chicago 311 Service Requests (~1.3 GB, 12 CSV fajlova, 4M+ redova)

---

## 1. Imports

In [4]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
import json
from pprint import pprint

import connection
import load_data
import base_queries
import optimized_queries
import optimize_schema
import query_executor

## 2. MongoDB konekcija

In [5]:
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DATABASE_NAME = "chicago_311_db"

client, database = connection.connect_to_mongodb(MONGO_URI, MONGO_DATABASE_NAME)

Povezan na MongoDB: mongodb://localhost:27017/
Baza podataka: chicago_311_db


## 3. Učitavanje podataka

Učitavanje 12 CSV fajlova u 6 normalizovanih kolekcija:
- `requests` - zajednička polja svih zahteva
- `locations` - deduplikovani geografski podaci
- `vehicle_details` - detalji napuštenih vozila
- `sanitation_details` - sanitarni detalji (glodari, đubre, kodne povrede)
- `environment_details` - infrastruktura i okruženje (rupe, svetla, grafiti, drveće)
- `building_details` - detalji napuštenih zgrada

In [7]:
load_data.drop_all_collections(database)
stats = load_data.load_all_data(database)

Sve kolekcije obrisane.

Ucitavanje: 311-service-requests-abandoned-vehicles.csv
  Redova u CSV: 242,734


  311-service-requests-abandoned: 100%|██████████| 242734/242734 [00:33<00:00, 7326.44it/s] 



Ucitavanje: 311-service-requests-alley-lights-out.csv
  Redova u CSV: 219,010


  311-service-requests-alley-lig: 100%|██████████| 219010/219010 [00:14<00:00, 15433.39it/s]



Ucitavanje: 311-service-requests-garbage-carts.csv
  Redova u CSV: 403,791


...


Ucitavanje: 311-service-requests-tree-trims.csv
  Redova u CSV: 362,957


  311-service-requests-tree-trim: 100%|██████████| 362957/362957 [00:26<00:00, 13847.89it/s]



Ucitavanje: 311-service-requests-vacant-and-abandoned-buildings-reported.csv
  Redova u CSV: 65,119


  311-service-requests-vacant-an: 100%|██████████| 65119/65119 [00:05<00:00, 11254.75it/s]



STATISTIKA UCITAVANJA:
  requests: 4,329,643 dokumenata
  locations: 934,715 dokumenata
  vehicle_details: 242,734 dokumenata
  sanitation_details: 875,642 dokumenata
  environment_details: 3,146,148 dokumenata
  building_details: 65,119 dokumenata


### Pregled broja dokumenata po kolekciji

In [9]:
print("Broj dokumenata po kolekciji:")
print("=" * 40)
for coll_name in ["requests", "locations", "vehicle_details",
                   "sanitation_details", "environment_details", "building_details"]:
    count = database[coll_name].count_documents({})
    print(f"  {coll_name}: {count:,}")

print(f"\nPrimer dokumenta iz 'requests':")
pprint(database["requests"].find_one())

print(f"\nPrimer dokumenta iz 'locations':")
pprint(database["locations"].find_one())

Broj dokumenata po kolekciji:
  requests: 4,329,643
  locations: 934,715
  vehicle_details: 242,734
  sanitation_details: 875,642
  environment_details: 3,146,148
  building_details: 65,119

Primer dokumenta iz 'requests':
{'_id': ObjectId('6a484aa126c0927df90c51e0'),
 'completion_date': datetime.datetime(2015, 4, 9, 0, 0),
 'creation_date': datetime.datetime(2015, 4, 8, 0, 0),
 'current_activity': 'FVI - Outcome',
 'location_id': ObjectId('6a484aa126c0927df90c51df'),
 'most_recent_action': 'Vehicle was moved from original address requested',
 'request_type': 'Abandoned Vehicle Complaint',
 'service_request_number': '15-01207496',
 'status': 'Completed'}

Primer dokumenta iz 'locations':
{'_id': ObjectId('6a484aa126c0927df90c51df'),
 'community_area': 6,
 'latitude': 41.93702589972641,
 'longitude': -87.64615132728282,
 'police_district': 19,
 'street_address': '3020 N WATERLOO CT',
 'ward': 44,
 'zip_code': '60657'}


## 4. Kreiranje indeksa za inicijalnu šemu

In [ ]:
load_data.create_base_indexes(database)

### Pregled kreiranih indeksa

In [ ]:
for coll_name in ["requests", "locations", "vehicle_details",
                   "sanitation_details", "environment_details", "building_details"]:
    indexes = database[coll_name].index_information()
    print(f"\n{coll_name}:")
    for idx_name, idx_info in indexes.items():
        print(f"  {idx_name}: {idx_info['key']}")

## 5. Analiza upita — inicijalna (normalizovana) šema

---

### Q1: Vreme rešavanja po vrsti prijave i gradskoj oblasti

Koje vrste prijava imaju najduže prosečno vreme rešavanja i kako se to razlikuje po gradskim oblastima (community area)?

**Poslovni značaj:** otkriva probleme koji najviše opterećuju gradske službe.

In [ ]:
pipeline = base_queries.query_1_resolution_by_type_area()
results_q1, time_q1 = query_executor.execute_query(
    database["requests"], pipeline, "Q1 - Vreme resavanja po tipu i oblasti")

df = pd.DataFrame(results_q1)
print(df.to_string(index=False))

### Q2: Zanemarene community areas

In [ ]:
pipeline = base_queries.query_2_neglected_areas()
results_q2, time_q2 = query_executor.execute_query(
    database["requests"], pipeline, "Q2 - Zanemarene community areas")

df = pd.DataFrame(results_q2)
print(df.to_string(index=False))

### Q3: Hotspot lokacije

Da li postoje lokacije (adrese) koje konstantno (kroz više godina) generišu veliki broj prijava i koje vrste problema u njima dominiraju?

**Poslovni značaj:** identifikacija problematičnih delova grada.

In [ ]:
pipeline = base_queries.query_3_hotspot_locations()
results_q3, time_q3 = query_executor.execute_query(
    database["requests"], pipeline, "Q3 - Hotspot lokacije")

df = pd.DataFrame(results_q3)
print(df.to_string(index=False))

### Q4: Problematični blokovi

In [ ]:
pipeline = base_queries.query_4_problem_blocks()
results_q4, time_q4 = query_executor.execute_query(
    database["requests"], pipeline, "Q4 - Problematicni blokovi")

df = pd.DataFrame(results_q4)
print(df.to_string(index=False))

### Q5: Najčešći tip prijave po gradskoj oblasti

Koji je najčešći tip prijave u svakoj gradskoj oblasti (community area) i koliki procenat svih prijava u toj oblasti predstavlja?

**Poslovni značaj:** šta je najčešći problem i koliko dominira u odnosu na ostale.

In [ ]:
pipeline = base_queries.query_5_top_types_per_area()
results_q5, time_q5 = query_executor.execute_query(
    database["requests"], pipeline, "Q5 - Najcesci tip po oblasti")

df = pd.DataFrame(results_q5)
print(df.to_string(index=False))

### Sačuvaj vremena izvršavanja base upita

In [ ]:
base_times = {
    "Q1": time_q1, "Q2": time_q2, "Q3": time_q3, "Q4": time_q4, "Q5": time_q5
}

print("\nVremena izvrsavanja (base sema):")
print("=" * 40)
for name, t in base_times.items():
    print(f"  {name}: {t:.3f}s")

## 6. Optimizacija šeme

Migracija 6 normalizovanih kolekcija → 1 kolekcija sa embedded dokumentima.  
Koristi cache pristup: učita sve lookup podatke u memoriju, pa gradi nove dokumente.

In [ ]:
optimize_schema.migrate_to_embedded(database)

In [ ]:
optimize_schema.create_optimized_indexes(database)

In [ ]:
print(f"Broj dokumenata u optimizovanoj kolekciji: {database['requests_optimized'].count_documents({}):,}")
print(f"\nPrimer embedded dokumenta:")
pprint(database["requests_optimized"].find_one())

### Pregled indeksa optimizovane kolekcije

In [ ]:
indexes = database["requests_optimized"].index_information()
print("Indeksi na 'requests_optimized':")
for idx_name, idx_info in indexes.items():
    print(f"  {idx_name}: {idx_info['key']}")

## 7. Optimizovani upiti (embedded šema, bez $lookup)

---

### Q1 (optimizovano): Vreme rešavanja po tipu i oblasti

In [ ]:
pipeline = optimized_queries.query_1_resolution_by_type_area()
opt_results_q1, opt_time_q1 = query_executor.execute_query(
    database["requests_optimized"], pipeline, "Q1-opt")

df = pd.DataFrame(opt_results_q1)
print(df.to_string(index=False))

### Q2 (optimizovano): Zanemarene community areas

In [ ]:
pipeline = optimized_queries.query_2_neglected_areas()
opt_results_q2, opt_time_q2 = query_executor.execute_query(
    database["requests_optimized"], pipeline, "Q2-opt")

df = pd.DataFrame(opt_results_q2)
print(df.to_string(index=False))

### Q3 (optimizovano): Hotspot lokacije

In [ ]:
pipeline = optimized_queries.query_3_hotspot_locations()
opt_results_q3, opt_time_q3 = query_executor.execute_query(
    database["requests_optimized"], pipeline, "Q3-opt")

df = pd.DataFrame(opt_results_q3)
print(df.to_string(index=False))

### Q4 (optimizovano): Problematični blokovi

In [ ]:
pipeline = optimized_queries.query_4_problem_blocks()
opt_results_q4, opt_time_q4 = query_executor.execute_query(
    database["requests_optimized"], pipeline, "Q4-opt")

df = pd.DataFrame(opt_results_q4)
print(df.to_string(index=False))

### Q5 (optimizovano): Najčešći tip po oblasti

In [ ]:
pipeline = optimized_queries.query_5_top_types_per_area()
opt_results_q5, opt_time_q5 = query_executor.execute_query(
    database["requests_optimized"], pipeline, "Q5-opt")

df = pd.DataFrame(opt_results_q5)
print(df.to_string(index=False))

## 8. Uporedna analiza performansi

In [ ]:
opt_times = {
    "Q1": opt_time_q1, "Q2": opt_time_q2, "Q3": opt_time_q3,
    "Q4": opt_time_q4, "Q5": opt_time_q5
}

comparison = []
for q in ["Q1", "Q2", "Q3", "Q4", "Q5"]:
    bt = base_times[q]
    ot = opt_times[q]
    improvement = ((bt - ot) / bt) * 100 if bt > 0 else 0
    comparison.append({
        "Upit": q,
        "Base (s)": round(bt, 3),
        "Optimized (s)": round(ot, 3),
        "Ubrzanje (%)": round(improvement, 1)
    })

df_comp = pd.DataFrame(comparison)
print(df_comp.to_string(index=False))

avg_improvement = df_comp["Ubrzanje (%)"].mean()
print(f"\nProsecno ubrzanje: {avg_improvement:.1f}%")

### Grafički prikaz poređenja

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

queries = list(base_times.keys())
base_vals = [base_times[q] for q in queries]
opt_vals = [opt_times[q] for q in queries]

x = range(len(queries))
width = 0.35

axes[0].bar([i - width/2 for i in x], base_vals, width, label='Base (normalizovana)', color='#e74c3c')
axes[0].bar([i + width/2 for i in x], opt_vals, width, label='Optimizovana (embedded)', color='#2ecc71')
axes[0].set_xlabel('Upit')
axes[0].set_ylabel('Vreme izvrsavanja (s, log skala)')
axes[0].set_title('Poredjenje vremena izvrsavanja')
axes[0].set_yscale('log')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(queries)
axes[0].legend()

speedups = [base_times[q] / opt_times[q] if opt_times[q] > 0 else 0 for q in queries]
axes[1].barh(queries, speedups, color='#3498db')
axes[1].set_xlabel('Faktor ubrzanja (x puta brze)')
axes[1].set_title('Faktor ubrzanja po upitu')

plt.tight_layout()
plt.savefig("performance_comparison_marko.png", dpi=150, bbox_inches='tight')
plt.show()

### Explain analiza

Poređenje plana izvršavanja (`explain`) za base i optimizovanu šemu — pokazuje `COLLSCAN` vs `IXSCAN` i korišćeni indeks.

In [ ]:
import json

def analyze_explain(collection_name, pipeline, label):
    explain = database.command("aggregate", collection_name,
        pipeline=pipeline, explain=True, allowDiskUse=True)

    print(f"\n{'=' * 60}")
    print(f"EXPLAIN: {label}")
    print(f"{'=' * 60}")

    stages = explain.get("stages", [])
    if stages:
        first = stages[0]
        cursor = first.get("$cursor", {})
        query_planner = cursor.get("queryPlanner", {})
        winning = query_planner.get("winningPlan", {})

        stage = winning.get("stage", "N/A")
        input_stage = winning.get("inputStage", {})
        index_name = input_stage.get("indexName", "N/A")

        print(f"  Winning plan stage: {stage}")
        if stage == "IXSCAN" or input_stage.get("stage") == "IXSCAN":
            print(f"  Index: {index_name}")
        elif stage == "COLLSCAN":
            print(f"  UPOZORENJE: Collection scan (nema indeksa)")
    else:
        print(json.dumps(explain, indent=2, default=str)[:500])

analyze_explain("requests",
    base_queries.query_5_top_types_per_area(),
    "Q5 Base (normalizovana)")
analyze_explain("requests_optimized",
    optimized_queries.query_5_top_types_per_area(),
    "Q5 Optimizovana (embedded)")

## Zaključak

Analiza Q1–Q5 pokriva rad gradskih službi iz perspektive **analitičara gradske službe**:

1. **Q1 — Vreme rešavanja po tipu i oblasti:** koje vrste prijava najviše opterećuju službe i kako se prosečno vreme rešavanja razlikuje po community area (najgora vs najbolja oblast).
2. **Q2 — Zanemarene community areas:** oblasti gde je prosečno vreme rešavanja infrastrukturnih problema više od 1.5× gradskog proseka.
3. **Q3 — Hotspot lokacije:** adrese koje kroz više godina konstantno generišu veliki broj prijava, uz dominantni tip problema.
4. **Q4 — Problematični blokovi:** adrese sa 5+ različitih tipova žalbi u jednoj godini.
5. **Q5 — Najčešći tip po oblasti:** dominantni tip prijave u svakoj community area i njegov procenat u ukupnom broju prijava.

**Napomena o skupu podataka:** dva prvobitno predložena pitanja nisu bila izvodljiva nad ovim (istorijskim) skupom podataka — analiza po **agencijama i predviđenom roku (SLA)** (skup ne sadrži polje o agenciji/odeljenju ni o roku) i analiza **po satu / „kritičnim terminima" tokom dana** (svi vremenski pečati su na `00:00:00`, bez informacije o satu). Za ta dva mesta zadržana su relevantna pitanja koja skup podataka podržava (Q2 i Q4).

**Optimizacija:** migracijom u embedded šemu eliminisani su `$lookup` operatori, a kompozitni indeksi (npr. `location.community_area + request_type` za Q5, `location.street_address` za Q3) omogućavaju `IXSCAN` umesto `COLLSCAN`, što je rezultovalo značajnim ubrzanjem svih upita.